# 05 - Feature Engineering


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 50)

from src.data.load_data import load_csv
from pipelines.data_pipeline import engineer_features
from src.utils.config import load_config, resolve_path
from src.utils.helpers import save_dataframe

config = load_config()
df_clean = load_csv(resolve_path(config['data']['interim_path']))
print(f"Cleaned dataset shape: {df_clean.shape}")

2026-09-20 22:31:10 | INFO     | src.data.load_data | Loaded CSV 'manufacturing_interim.csv' with shape (3240, 17)


Cleaned dataset shape: (3240, 17)


## Applying the feature engineering function



In [2]:
df_processed = engineer_features(df_clean, config)
print(f"Shape before feature engineering: {df_clean.shape}")
print(f"Shape after feature engineering:  {df_processed.shape}")
new_columns = [c for c in df_processed.columns if c not in df_clean.columns]
print(f"New columns added: {new_columns}")

2026-09-20 22:31:12 | INFO     | pipelines.data_pipeline | Feature engineering complete: added CostPerUnit, QualityCategory, DowntimeCategory, DefectFlagLabel.


Shape before feature engineering: (3240, 17)
Shape after feature engineering:  (3240, 21)
New columns added: ['CostPerUnit', 'QualityCategory', 'DowntimeCategory', 'DefectFlagLabel']


In [3]:
df_processed[['ProductionCost', 'ProductionVolume', 'CostPerUnit', 'QualityScore', 'QualityCategory', 'DowntimePercentage', 'DowntimeCategory', 'DefectStatus', 'DefectFlagLabel']].head(10)

,ProductionCost,ProductionVolume,CostPerUnit,QualityScore,QualityCategory,DowntimePercentage,DowntimeCategory,DefectStatus,DefectFlagLabel
0,13175.403783,202,65.22,63.463494,Poor,0.052343,Low,1,Defective
1,19770.046093,535,36.95,83.697818,Average,4.908328,High,1,Defective
2,19060.820997,960,19.86,90.350550,Good,2.464923,Medium,1,Defective
3,5647.606037,370,15.26,67.628690,Poor,4.692476,High,1,Defective
4,7472.222236,206,36.27,82.728334,Average,2.746726,Medium,1,Defective
5,6975.931602,171,40.79,92.568436,Good,3.027324,Medium,1,Defective
6,15889.698650,800,19.86,90.729911,Good,3.559561,High,1,Defective
7,17266.779948,120,143.89,92.119681,Good,1.604879,Medium,1,Defective
8,8202.670495,714,11.49,95.172937,Good,3.494920,Medium,1,Defective
9,12587.790394,221,56.96,97.507284,Good,2.633960,Medium,0,Not Defective


## Checking the new categorical bands

We verify the bucket counts make sense (no empty or dominant-to-the-point-of-uselessness categories).

In [4]:
df_processed['QualityCategory'].value_counts()

QualityCategory
Good       1236
Average    1205
Poor        799
Name: count, dtype: int64

In [5]:
df_processed['DowntimeCategory'].value_counts()

DowntimeCategory
Medium    1278
Low        982
High       980
Name: count, dtype: int64

**Interpretation:** Both new categorical features produce a reasonable, non-degenerate split of records across their bands, confirming the configured thresholds in `config/config.yaml` are sensible for this dataset.

## Saving the processed dataset

The final, feature-engineered dataset is saved to `data/processed/`, which is the input for the business analysis notebook and the dashboards.

In [6]:
processed_path = resolve_path(config['data']['processed_path'])
save_dataframe(df_processed, processed_path)
print(f"Processed dataset saved to: {processed_path}")

Processed dataset saved to: E:\Python projects\manufacturing-quality-operational-performance-analysis\data\processed\manufacturing_processed.csv


## Summary

Four engineered features were added on top of the cleaned dataset: `CostPerUnit`, `QualityCategory`, `DowntimeCategory`, and `DefectFlagLabel`.